# US-008：公开数据集端到端实战

**目标：** 在真实公开 EEG 数据集上跑通完整分析 pipeline，产出可复用的 Notebook。

**推荐数据集：**
- **BCI Competition IV Dataset 2a**：四类运动想象，22 导 EEG，9 个被试
- **OpenNeuro ds003505**：听觉 oddball，简单易懂
- **PhysioNet EEG Motor Movement/Imagery**：运动/想象任务，64 导 EEG

## 1. 选择数据集

本 notebook 以 **BCI Competition IV Dataset 2a** 为例。

### 数据集简介
- 任务：四类运动想象（左手、右手、双脚、舌）
- 通道：22 导 EEG + 3 导 EOG
- 采样率：250 Hz
- 被试：9 人
- 数据格式：GDF

## 2. 安装依赖

```bash
pip install mne moabb pooch
```

MOABB 是 MNE 生态中自动下载公开 BCI 数据集的工具。

In [ ]:
# ===== 方式一：用 MOABB 自动下载（推荐）=====
# from moabb.datasets import BNCI2014_001
# dataset = BNCI2014_001()
# dataset.download()

# ===== 方式二：直接下载 .gdf 文件 =====
# wget https://lampx.tugraz.at/~bci/competition/iv/2a/A01T.gdf

print("请选择一种方式下载数据。以下代码假设数据在 ./data/BCICIV_2a/ 目录下。")

## 3. 加载数据

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ===== 路径配置 =====
data_dir = Path("./data/BCICIV_2a/")
subject = 1  # 被试编号 1-9
session = "T"  # T=训练, E=测试

# ===== 读取 GDF 文件 =====
# 文件名格式：A{sub:02d}{session}.gdf
gdf_path = data_dir / f"A{subject:02d}{session}.gdf"
# raw = mne.io.read_raw_gdf(gdf_path, preload=True)
# print(raw)
# print(f"通道名: {raw.ch_names}")

print("实际运行时取消注释即可。")

## 4. 配置通道与蒙太奇

In [ ]:
# ===== BCIC IV 2a 的 22 个 EEG 通道（标准 10-20 子集）=====
ch_names_bci = [
    'Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
    'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6',
    'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
    'P1', 'Pz', 'P2', 'POz'
]

# ===== 示例：设置通道类型和蒙太奇 =====
# raw.set_channel_types({ch: 'eeg' for ch in ch_names_bci})
# raw.set_channel_types({'EOG-left': 'eog', 'EOG-central': 'eog', 'EOG-right': 'eog'})

# 设置标准 10-20 蒙太奇
# montage = mne.channels.make_standard_montage('standard_1020')
# raw.set_montage(montage)

print("取消注释运行。")

## 5. 事件提取

BCIC IV 2a 的事件 ID：
- 769 = 左手想象
- 770 = 右手想象
- 771 = 双脚想象
- 772 = 舌头想象

In [ ]:
# ===== 提取事件 =====
# events = mne.find_events(raw, stim_channel='auto', shortest_event=1)
# print(f"事件 ID: {np.unique(events[:, 2])}")

event_id_bci = {
    'left_hand': 769,
    'right_hand': 770,
    'feet': 771,
    'tongue': 772,
}

# 如果只分析两类（左手 vs 右手），可以只选两个
event_id_subset = {'left_hand': 769, 'right_hand': 770}
print(f"事件映射: {event_id_bci}")

## 6. 完整 Pipeline

把 US-001 ~ US-007 学的全部串起来。

In [ ]:
# ===== 导入我们之前的工具函数 =====
import sys
sys.path.insert(0, '../../scripts')

from us004_preprocessing import preprocess_pipeline
from us005_ica import ica_pipeline
from us006_epoching import create_epochs, equalize_conditions
from us007_viz_tfr import compute_tfr, plot_tfr, plot_compare

print("工具函数已导入。")

In [ ]:
# ===== 完整 Pipeline（伪代码）=====

# Step 1: 加载
# raw = mne.io.read_raw_gdf(gdf_path, preload=True)

# Step 2: 配置通道
# raw.set_channel_types({...})
# raw.set_montage(montage)

# Step 3: 预处理
# raw_clean = preprocess_pipeline(
#     raw, notch_freqs=50, l_freq=1, h_freq=40,
#     target_sfreq=150, ref_channels='average'
# )

# Step 4: ICA 去眼电
# raw_ica, ica = ica_pipeline(raw_clean, n_components=20)

# Step 5: 事件提取
# events = mne.find_events(raw_ica, stim_channel='auto')

# Step 6: Epoching
# epochs = create_epochs(
#     raw_ica, events, event_id_subset,
#     tmin=-0.5, tmax=2.5,
#     baseline=(-0.5, 0),
#     reject=dict(eeg=150e-6),
# )

# Step 7: 条件对比
# evoked_L = epochs['left_hand'].average()
# evoked_R = epochs['right_hand'].average()
# plot_compare({'Left Hand': evoked_L, 'Right Hand': evoked_R})

# Step 8: 时频分析（alpha/beta ERD/ERS）
# power_L = compute_tfr(epochs['left_hand'], decim=3)
# plot_tfr(power_L, picks='C3', baseline=(-0.5, 0), mode='logratio')

print("以上为 Pipeline 骨架，填入实际数据路径即可运行。")

## 7. 运动想象 ERD/ERS 分析

运动想象的经典发现：
- **对侧 ERD**：想象右手时，左侧感觉运动区（C3）alpha/beta 功率下降
- **同侧 ERS**：同侧区域可能出现轻微功率上升

### 关键通道
- C3：右侧运动（左手想象时 ERD）
- C4：左侧运动（右手想象时 ERD）
- Cz：双脚运动想象

### 关键频段
- **mu 节律 (8-13 Hz)**：alpha 频段的感觉运动节律
- **beta (13-30 Hz)**：运动准备和执行相关

In [ ]:
# ===== ERD/ERS 计算示例 =====

# 定义 alpha 和 beta 频段
freqs_alpha = np.arange(8, 14)   # 8-13 Hz
freqs_beta = np.arange(14, 31)   # 14-30 Hz

# ===== 伪代码 =====
# # 计算左手想象的 TFR
# power_L = tfr_morlet(
#     epochs['left_hand'],
#     freqs=np.logspace(*np.log10([4, 35]), num=30),
#     n_cycles=freqs / 2,
#     decim=3,
# )

# # alpha 频段平均
# erd_alpha_L = power_L.copy().crop(fmin=8, fmax=13).data.mean(axis=2)

# # 绘制 C3 vs C4 的 alpha ERD 时间曲线
# # ...

print("ERD/ERS 分析代码框架已就绪。")

## 8. 导出结果

把分析结果保存为可复用的格式。

In [ ]:
# 保存关键中间结果
# raw_clean.save('sub01_clean.fif', overwrite=True)
# epochs.save('sub01_epochs.fif', overwrite=True)
# evoked_L.save('sub01_evoked_left.fif', overwrite=True)

# 导出 NumPy 数组供后续用 ML 分析
# X = epochs.get_data()                # (n_trials, n_channels, n_times)
# y = epochs.events[:, 2]              # 事件标签
# np.save('sub01_X.npy', X)
# np.save('sub01_y.npy', y)

print("导出方法就绪。")

## 9. 其他推荐数据集

| 数据集 | 来源 | 范式 | 难度 |
|--------|------|------|------|
| BCIC IV 2a | MOABB | 四类运动想象 | ⭐⭐ |
| BCIC IV 2b | MOABB | 两类运动想象 | ⭐ |
| PhysioNet MI | PhysioNet | 运动/想象 | ⭐⭐ |
| OpenNeuro ds003505 | OpenNeuro | 听觉 oddball | ⭐ |
| ERP CORE | ERP CORE | 六种 ERP 范式 | ⭐⭐ |
| MNE sample | MNE | 听觉+视觉 | ⭐ |

## 10. 产出检查清单

- [ ] 数据成功加载
- [ ] 预处理 pipeline 跑通
- [ ] ICA 去眼电完成并验证
- [ ] Epochs 成功切分
- [ ] ERP 波形 + 地形图产出
- [ ] 时频图产出
- [ ] 条件间对比完成
- [ ] 代码整理为可复用函数
- [ ] Notebook 从头到尾可运行